# Skin Cancer Classifier — DenseNet121 Binary (Benign / Malignant)

Clean binary classification pipeline on ISIC data.

| Section | Content |
|---|---|
| 1. Preprocessing | Read `diagnosis_3`, map Nevus→Benign / else→Malignant, stratified 70/15/15 split |
| 2. Dataset | `SkinLesionDataset` with augmentation for train, plain normalisation for val/test |
| 3. Model | Pretrained DenseNet121, head replaced with `Linear(1024→256)→ReLU→Dropout→Linear(256→2)` |
| 4. Training | Phase 1 frozen backbone, Phase 2 full fine-tune; early stopping on val binary AUC |
| 5. Evaluation | Binary AUC, confusion matrix, malignant sensitivity & specificity |
| 6. FastAPI | Inference endpoint for deployment |

**Data**: `/kaggle/working/isic_data/`  — flat image folder + `metadata.csv` with `diagnosis_3` column.

In [ ]:
import os
import io
import csv
import time
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import DenseNet121_Weights
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS_1 = 5
EPOCHS_2 = 15
LR_1 = 1e-3
LR_2 = 1e-5
PATIENCE = 5
NUM_CLASSES = 2
CLASSES = ['Benign', 'Malignant']
LABEL2IDX = {'Benign': 0, 'Malignant': 1}

ISIC_DIR = Path('/kaggle/working/isic_data')
SPLITS_DIR = Path('/kaggle/working/splits')
WORKING_DIR = Path('/kaggle/working')
BEST_MODEL_PATH = WORKING_DIR / 'best_model.pth'

SPLITS_DIR.mkdir(exist_ok=True)
(WORKING_DIR / 'checkpoints').mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Preprocessing — Stratified Splits (70 / 15 / 15)

- Reads `metadata.csv` from `/kaggle/working/isic_data/`
- Applies binary label mapping via `diagnosis_3`: `Nevus` → Benign (0), everything else → Malignant (1)
- Performs stratified split and writes `train.csv`, `val.csv`, `test.csv` to `/kaggle/working/splits/`

In [ ]:
def collect_samples(isic_dir: Path) -> list:
    csv_files = sorted(isic_dir.glob('*.csv'))
    if not csv_files:
        raise RuntimeError(f'No CSV found in {isic_dir}')
    meta_csv = csv_files[0]
    print(f'Reading: {meta_csv.name}')

    df = pd.read_csv(meta_csv)
    print(f'Rows: {len(df)}  |  Columns: {list(df.columns)}')

    # Binary label: Nevus = Benign (0), everything else = Malignant (1)
    df['label'] = df['diagnosis_3'].apply(lambda x: 0 if x == 'Nevus' else 1)
    print('Label distribution:')
    print(df['label'].value_counts().rename({0: 'Benign', 1: 'Malignant'}).to_string())

    samples = []
    missing = 0
    for _, row in df.iterrows():
        isic_id = str(row.get('isic_id', row.get('image', ''))).strip()
        label_idx = int(row['label'])
        for ext in ('.jpg', '.JPG', '.jpeg', '.png'):
            img_path = isic_dir / f'{isic_id}{ext}'
            if img_path.exists():
                samples.append((str(img_path), label_idx))
                break
        else:
            missing += 1
    if missing:
        print(f'Warning: {missing} rows had no matching image.')
    return samples


def stratified_split(samples: list, train_frac=0.70, val_frac=0.15, seed=SEED):
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for s in samples:
        by_class[s[1]].append(s)

    train, val, test = [], [], []
    for label_idx, cls_samples in sorted(by_class.items()):
        rng.shuffle(cls_samples)
        n = len(cls_samples)
        n_train = max(1, round(n * train_frac))
        n_val = max(1, round(n * val_frac))
        n_train = min(n_train, n - 2) if n >= 3 else n_train
        train.extend(cls_samples[:n_train])
        val.extend(cls_samples[n_train:n_train + n_val])
        test.extend(cls_samples[n_train + n_val:])
    return train, val, test


def write_split(split: list, path: Path) -> None:
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['image_path', 'label_idx', 'label_name'])
        for img_path, label_idx in split:
            writer.writerow([img_path, label_idx, CLASSES[label_idx]])


def print_split_stats(name: str, split: list) -> None:
    counts = defaultdict(int)
    for _, idx in split:
        counts[idx] += 1
    print(f'  {name} ({len(split)} total) — ' +
          ', '.join(f'{CLASSES[k]}: {counts[k]}' for k in sorted(counts)))


# ── Run ───────────────────────────────────────────────────────────────────────
print(f'Scanning {ISIC_DIR} ...')
_samples = collect_samples(ISIC_DIR)
if not _samples:
    raise RuntimeError(f'No samples found in {ISIC_DIR}.')
print(f'Total matched: {len(_samples)}')

_train, _val, _test = stratified_split(_samples)
print('\nSplit sizes:')
for _name, _split in [('train', _train), ('val', _val), ('test', _test)]:
    print_split_stats(_name, _split)

write_split(_train, SPLITS_DIR / 'train.csv')
write_split(_val,   SPLITS_DIR / 'val.csv')
write_split(_test,  SPLITS_DIR / 'test.csv')
print(f'\nSplits saved to {SPLITS_DIR}')

## 2. Dataset & Transforms

Training augmentation: random crop, horizontal/vertical flip, rotation ±20°, colour jitter, ImageNet normalisation.

Val / test: resize 256 → centre crop 224 → ImageNet normalisation.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224


def get_transforms(split: str) -> transforms.Compose:
    if split == 'train':
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class SkinLesionDataset(Dataset):
    def __init__(self, csv_path: Path, split: str = 'train') -> None:
        self.transform = get_transforms(split)
        self.samples = []
        with open(csv_path, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                self.samples.append((row['image_path'], int(row['label_idx'])))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        return self.transform(image), label


print('SkinLesionDataset defined.')

## 3. Model — DenseNet121

```
DenseNet121 backbone (pretrained ImageNet)
    → Linear(1024, 256) → ReLU → Dropout(0.4) → Linear(256, 2)
```

Phase 1: only the head is trainable (`freeze_backbone`).
Phase 2: full network (`unfreeze_all`).

In [ ]:
def build_model(num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    weights = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights)
    in_features = model.classifier.in_features  # 1024
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(256, num_classes),
    )
    return model


def freeze_backbone(model: nn.Module) -> None:
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith('classifier')


def unfreeze_all(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_m = build_model(pretrained=False)
freeze_backbone(_m)
print(f'Trainable (frozen backbone): {count_trainable(_m):,}')
unfreeze_all(_m)
print(f'Trainable (all unfrozen):    {count_trainable(_m):,}')
del _m

## 4. Training

**Phase 1** (epochs=5): frozen backbone, Adam LR=1e-3, ReduceLROnPlateau.

**Phase 2** (epochs=15): full fine-tune, Adam LR=1e-5, CosineAnnealingLR.

Class weights computed with `sklearn.utils.class_weight.compute_class_weight`.
Early stopping on val binary AUC (patience=5). Best model → `/kaggle/working/best_model.pth`.

In [ ]:
def make_loader(csv_path: Path, split: str) -> DataLoader:
    ds = SkinLesionDataset(csv_path, split=split)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )


def compute_macro_auc(model: nn.Module, loader: DataLoader) -> float:
    """Binary AUC using the Malignant (class 1) probability score."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            probs = torch.softmax(model(images), dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    all_probs  = np.concatenate(all_probs,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    try:
        return float(roc_auc_score(all_labels, all_probs[:, 1]))
    except ValueError:
        return 0.0


def train_one_epoch(model, loader, optimizer, criterion) -> float:
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def save_checkpoint(model, epoch: int, val_auc: float, path: Path) -> None:
    torch.save({
        'epoch': epoch,
        'val_auc': val_auc,
        'model_state_dict': model.state_dict(),
        'classes': CLASSES,
    }, path)
    print(f'  Saved → {path}  (val_auc={val_auc:.4f})')


def run_phase(phase, model, train_loader, val_loader, optimizer, scheduler,
              criterion, num_epochs, log_writer) -> float:
    best_auc, no_improve = 0.0, 0
    phase_path = WORKING_DIR / 'checkpoints' / f'best_phase{phase}.pth'

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_auc    = compute_macro_auc(model, val_loader)
        elapsed    = time.time() - t0

        print(f'  Phase {phase} | Epoch {epoch:>3}/{num_epochs} '
              f'| loss {train_loss:.4f} | val_auc {val_auc:.4f} | {elapsed:.1f}s')
        log_writer.writerow([phase, epoch, f'{train_loss:.6f}', f'{val_auc:.6f}'])

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_auc)
        else:
            scheduler.step()

        if val_auc > best_auc:
            best_auc, no_improve = val_auc, 0
            save_checkpoint(model, epoch, val_auc, phase_path)
            save_checkpoint(model, epoch, val_auc, BEST_MODEL_PATH)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping after {PATIENCE} non-improving epochs.')
                break

    return best_auc


# ── Build loaders ─────────────────────────────────────────────────────────────
train_loader = make_loader(SPLITS_DIR / 'train.csv', 'train')
val_loader   = make_loader(SPLITS_DIR / 'val.csv',   'val')

# Class weights via sklearn (balanced)
_all_labels = np.array([s[1] for s in train_loader.dataset.samples])
_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=_all_labels)
class_weights = torch.tensor(_weights, dtype=torch.float).to(DEVICE)
print(f'Class weights — Benign: {_weights[0]:.4f}, Malignant: {_weights[1]:.4f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
model = build_model(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)

# ── Log file ──────────────────────────────────────────────────────────────────
_log_path = WORKING_DIR / 'training_log.csv'
_log_file = open(_log_path, 'w', newline='')
_log_writer = csv.writer(_log_file)
_log_writer.writerow(['phase', 'epoch', 'train_loss', 'val_auc'])

# ── Phase 1 ───────────────────────────────────────────────────────────────────
print('\n=== Phase 1: head only (backbone frozen) ===')
freeze_backbone(model)
optimizer1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_1
)
scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode='max', factor=0.5, patience=2
)
run_phase(1, model, train_loader, val_loader,
          optimizer1, scheduler1, criterion, EPOCHS_1, _log_writer)

# ── Phase 2 ───────────────────────────────────────────────────────────────────
print('\n=== Phase 2: full fine-tune ===')
unfreeze_all(model)
optimizer2 = torch.optim.Adam(model.parameters(), lr=LR_2, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=EPOCHS_2, eta_min=1e-7
)
run_phase(2, model, train_loader, val_loader,
          optimizer2, scheduler2, criterion, EPOCHS_2, _log_writer)

_log_file.close()
print(f'\nTraining complete. Log → {_log_path}')

## 5. Evaluation

Runs on the held-out test set using the best saved checkpoint:

- Binary AUC-ROC
- Confusion matrix (Benign / Malignant) → `/kaggle/working/confusion_matrix.png`
- Malignant sensitivity (recall) & specificity
- ROC curve → `/kaggle/working/roc_malignant.png`

In [ ]:
def load_best_model(ckpt_path: Path) -> nn.Module:
    m = build_model(num_classes=NUM_CLASSES, pretrained=False)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    m.load_state_dict(ckpt['model_state_dict'])
    m.to(DEVICE).eval()
    print(f"Loaded checkpoint  epoch={ckpt.get('epoch', '?')}  "
          f"val_auc={ckpt.get('val_auc', 0):.4f}")
    return m


@torch.no_grad()
def run_inference(m: nn.Module, loader: DataLoader):
    all_probs, all_preds, all_labels = [], [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = m(images)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())
    return (np.concatenate(all_probs,  axis=0),
            np.concatenate(all_preds,  axis=0),
            np.concatenate(all_labels, axis=0))


def report_binary_auc(all_probs, all_labels) -> None:
    score = roc_auc_score(all_labels, all_probs[:, 1])
    print(f'\n=== Binary AUC-ROC ===')
    print(f'  Malignant AUC: {score:.4f}')


def plot_confusion_matrix(all_preds, all_labels) -> None:
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, colorbar=True)
    ax.set_title('Confusion Matrix — Test Set')
    fig.tight_layout()
    out = WORKING_DIR / 'confusion_matrix.png'
    fig.savefig(out, dpi=150)
    plt.show()
    print(f'Saved → {out}')


def plot_roc_malignant(all_probs, all_labels) -> None:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
    roc_score = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2, label=f'AUC = {roc_score:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
           title='ROC Curve — Malignant')
    ax.legend(loc='lower right')
    fig.tight_layout()
    out = WORKING_DIR / 'roc_malignant.png'
    fig.savefig(out, dpi=150)
    plt.show()
    print(f'Saved → {out}')


def malignant_sens_spec(all_preds, all_labels):
    TP = int(((all_labels == 1) & (all_preds == 1)).sum())
    TN = int(((all_labels == 0) & (all_preds == 0)).sum())
    FP = int(((all_labels == 0) & (all_preds == 1)).sum())
    FN = int(((all_labels == 1) & (all_preds == 0)).sum())
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    return sensitivity, specificity


# ── Run ───────────────────────────────────────────────────────────────────────
test_ds = SkinLesionDataset(SPLITS_DIR / 'test.csv', split='test')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'Test set: {len(test_ds)} images')

eval_model = load_best_model(BEST_MODEL_PATH)
all_probs, all_preds, all_labels = run_inference(eval_model, test_loader)

report_binary_auc(all_probs, all_labels)
plot_confusion_matrix(all_preds, all_labels)
plot_roc_malignant(all_probs, all_labels)

sensitivity, specificity = malignant_sens_spec(all_preds, all_labels)
print(f'\n=== Malignant ===')
print(f'  Sensitivity (recall): {sensitivity:.4f}')
print(f'  Specificity:          {specificity:.4f}')

## 6. FastAPI Inference Endpoint

Deploy after training. Copy to `inference_api.py` and run:

```bash
CHECKPOINT_PATH=/kaggle/working/best_model.pth \
    uvicorn inference_api:app --host 0.0.0.0 --port 8000
```

```bash
curl -X POST "http://localhost:8000/predict" -F "file=@lesion.jpg"
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FastAPI inference endpoint — for deployment after training     ║
# ║  pip install fastapi uvicorn python-multipart                   ║
# ╚══════════════════════════════════════════════════════════════════╝

try:
    from fastapi import FastAPI, File, HTTPException, UploadFile
    from pydantic import BaseModel as _BaseModel
    _FASTAPI_AVAILABLE = True
except ImportError:
    _FASTAPI_AVAILABLE = False
    print('fastapi not installed — skipping.')

if _FASTAPI_AVAILABLE:
    _CKPT_PATH = Path(os.getenv('CHECKPOINT_PATH', str(BEST_MODEL_PATH)))
    _preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    app = FastAPI(
        title='Skin Lesion Classifier',
        description='DenseNet121 binary classifier — Benign / Malignant.',
        version='1.0.0',
    )

    _api_model = None

    @app.on_event('startup')
    def _startup():
        global _api_model
        if not _CKPT_PATH.exists():
            print(f'WARNING: checkpoint not found at {_CKPT_PATH}.')
            return
        _api_model = load_best_model(_CKPT_PATH)
        print('Model ready.')

    class PredictionResponse(_BaseModel):
        predicted_class: str
        predicted_index: int
        probabilities: dict

    class HealthResponse(_BaseModel):
        status: str
        model_loaded: bool
        device: str

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model_loaded=_api_model is not None,
                              device=str(DEVICE))

    @app.post('/predict', response_model=PredictionResponse)
    async def predict(file: UploadFile = File(...)):
        if _api_model is None:
            raise HTTPException(503, f'Model not loaded ({_CKPT_PATH}).')
        if file.content_type not in ('image/jpeg', 'image/png', 'image/jpg'):
            raise HTTPException(415, f'Unsupported type: {file.content_type}.')
        raw = await file.read()
        if not raw:
            raise HTTPException(400, 'Empty file.')
        try:
            img = Image.open(io.BytesIO(raw)).convert('RGB')
        except Exception as exc:
            raise HTTPException(400, f'Cannot decode image: {exc}') from exc
        tensor = _preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(_api_model(tensor), dim=1).squeeze(0).cpu().tolist()
        pred_idx = int(np.argmax(probs))
        return PredictionResponse(
            predicted_class=CLASSES[pred_idx],
            predicted_index=pred_idx,
            probabilities={c: round(p, 6) for c, p in zip(CLASSES, probs)},
        )

    print('FastAPI app defined. Run: uvicorn <module>:app --port 8000')